In [1]:
import pandas as pd
from vae_module import train_vae, encode, decode, load_vae
import numpy as np

In [2]:
wti_df = pd.read_csv('../data/wti_prices.csv', index_col=0, parse_dates=['Date'])
wti_df = np.log(wti_df).diff().dropna()
cols = list(wti_df.columns)
print(wti_df.shape)
print(cols)

(480, 1)
['WTI_price']


In [3]:
tb3ms_df = pd.read_csv('../data/tb3ms.csv', index_col=0, parse_dates=['date'])
cols = list(tb3ms_df.columns)
print(tb3ms_df.shape)
print(cols)

(481, 1)
['TB3MS']


In [4]:
# Combine TB3MS and WTI data by month (index)
combined_df = tb3ms_df.join(wti_df, how='inner').sort_index()

# Optional: set a clear index name
combined_df.index.name = "month"

print(combined_df.shape)
display(combined_df.tail())

(480, 2)


,TB3MS,WTI_price
month,,
2025-09-01,-0.20,-0.013973
2025-10-01,-0.10,-0.049189
2025-11-01,-0.04,-0.013725
2025-12-01,-0.19,-0.035418
2026-01-01,-0.02,0.035085


In [5]:
from data_pipeline import build_pipeline, inverse_transform, load_scaler

# Build windowed dataset from raw training DataFrame
windows, scaler = build_pipeline(
    df=combined_df,            # rows = timesteps, columns = variables
    scaler_save_path="scaler.pkl"
)
# windows shape: (num_windows, window_length, num_variables)

[Pipeline] Observations: 480 | Variables: 2
[Pipeline] Columns: ['TB3MS', 'WTI_price']
[Pipeline] Window length: 24 | Stride: 1
[Pipeline] Windows to be created: 457
[Pipeline] Scaler: minmax
------------------------------------------------------------
[Pipeline] Normalisation complete.
[Pipeline] Data range after scaling: [0.0000, 1.0000]
[Pipeline] Windows created: 457
[Pipeline] Output shape: (457, 24, 2)  (num_windows, window_length, num_variables)
[Pipeline] Scaler saved to: scaler.pkl
------------------------------------------------------------
[Pipeline] Pipeline complete. Ready for Module 2 (VAE).


In [6]:
windows.shape

(457, 24, 2)

In [19]:
# Train
from vae_module import VAEConfig

config = VAEConfig(
    num_epochs=300,
    latent_dim=1,             # force real compression
    kl_warmup_epochs=5,
    kl_weight_max=2.0,
    learning_rate=1e-4,
    verbose_every=25,
)

model, history = train_vae(
    data=windows,        # shape: (num_windows, window_len, num_variables)
    config=config,
    save_path="vae_checkpoint.pt"
)

# Encode new data
latent_sequences = encode(windows, model)

# Decode latent sequences back to data space
reconstructed = decode(latent_sequences, model)

# Load a saved model
model = load_vae("vae_checkpoint.pt")


[VAE] Device: cpu
[VAE] Input dim: 2 | Latent dim: 1 | Hidden dim: 16
[VAE] Windows: 457 | Window length: 24 | Variables: 2
[VAE] Train windows: 389 | Val windows: 68
[VAE] KL warmup: 5 epochs | Max KL weight: 2.0
------------------------------------------------------------
[VAE] Epoch    1/300 | KL weight: 0.4000 | Train — Total: 0.084296, Recon: 0.074603, KL: 0.024235 | Val   — Total: 0.080862, Recon: 0.071560, KL: 0.023255
[VAE] Epoch   25/300 | KL weight: 2.0000 | Train — Total: 0.039757, Recon: 0.033349, KL: 0.003204 | Val   — Total: 0.040077, Recon: 0.034144, KL: 0.002966
[VAE] Epoch   50/300 | KL weight: 2.0000 | Train — Total: 0.018041, Recon: 0.017821, KL: 0.000110 | Val   — Total: 0.020140, Recon: 0.019910, KL: 0.000115
[VAE] Epoch   75/300 | KL weight: 2.0000 | Train — Total: 0.012684, Recon: 0.012611, KL: 0.000037 | Val   — Total: 0.017040, Recon: 0.016887, KL: 0.000076
[VAE] Epoch  100/300 | KL weight: 2.0000 | Train — Total: 0.009914, Recon: 0.009879, KL: 0.000017 | Val  

In [21]:
import numpy as np

latent = encode(windows, model)
latent_flat = latent.reshape(-1, latent.shape[-1])

print("Latent mean per dimension:", latent_flat.mean(axis=0).round(4))
print("Latent std per dimension: ", latent_flat.std(axis=0).round(4))
print("Min std:", latent_flat.std(axis=0).min().round(6))

Latent mean per dimension: [-0.0002]
Latent std per dimension:  [0.0018]
Min std: 0.001789


In [2]:
# Train
from timegan_module import train_timegan, generate, load_timegan, TimeGANConfig

config = TimeGANConfig(
    hidden_dim=24,
    num_layers=1,
    noise_dim=2,
    num_epochs=200,
    batch_size=32,
    learning_rate=1e-3,
    early_stopping_patience=3,
    eval_every_n_epochs=40,
    verbose_every=20,
    random_seed=42,
    device="cuda", ##need to change to "cpu" if no GPU available
)

model, history = train_timegan(
    data=windows,               # shape: (num_windows, window_length, num_variables)
    save_path="timegan_checkpoint.pt"
)

# Generate synthetic windows
synthetic = generate(n_samples=200, model=model)
# synthetic shape: (200, window_length, num_variables)

# Load saved model
model = load_timegan("timegan_checkpoint.pt")

NameError: name 'windows' is not defined